# Анализ пользовательского поведения в сервисе «Яндекс Книги» 

## Часть 1. Проверка гипотезы в Python и составление аналитической записки

## 1. Введение

### 1.1 Цели и задачи проекта

Цель проекта — изучить поведение пользователей сервиса Bookmate в Москве и Санкт-Петербурге и определить, есть ли различия в их активности и количестве часов прослушивания.

Задачи проекта:

изучить и подготовить данные о пользователях;  
проверить данные на дубликаты и пересечения между группами и обработать их;  
сравнить размеры групп пользователей из Москвы и Санкт-Петербурга;  
рассчитать основные статистические показатели по количеству часов прослушивания;  
выбрать статистический тест и сравнить группы с его помощью;  
определить, являются ли различия между группами статистически значимыми;  
сформулировать выводы по результатам анализа. 

### 1.2 Описание данных

yandex_knigi_data.csv - таблица с данными о пользователях из Москвы и Санкт-Петербурга с суммой часов их активности в приложении Яндекс Книги  

поля:  
city — город пользователя;  
puid — идентификатор пользователя;  
hours — общее количество часов активности.

###  1.3 Содержимое проекта

- Часть 1. Проверка гипотезы в Python и составление аналитической записки
    - 1. Введение
    - 2. Загрузка данных и знакомство с ними
    - 3. Проверка гипотезы в Python
    - 4. Аналитическая записка
- Часть 2. Анализ результатов A/B-тестирования
    - 1. Цель и задачи исследования
    - 2. Загрузка данных, оценка их целостности
    - 3.  По таблице `participants` оценка корректности проведения теста
    - 4. Оценка результатов A/B-тестирования и выводы


## 2. Загрузка данных и знакомство с ними

Загрузите данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
# импортируем нужные библиотеки
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu

In [2]:
# выгрузим данные
df = pd.read_csv("/datasets/yandex_knigi_data.csv")

In [3]:
df.head()

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


In [4]:
# удалим ненужный столбец Unnamed
df = df.drop(columns="Unnamed: 0")

In [5]:
# проверим полные дубликаты и отдельно дубликаты по пользоваьелям
df.duplicated().sum()

0

In [6]:
df["puid"].duplicated().sum()

244

In [7]:
user_groups = df.groupby("puid")["city"].nunique()

print("Пользователей в двух городах:", (user_groups > 1).sum())

Пользователей в двух городах: 244


Полных дуликатов нет, но при этом есть 244 дубликата по пользователям. Это означает что один и тот же пользователь пользовался сервисом и в Мск и в С-Пб. Для дальнейшего проведения a/b - теста нам нужны две непересекающиеся группы, поэтому 244 пользователя будут удалены из обеих групп.

In [8]:
# Находим пользователей, которые есть в обеих группах
both_cities = (
    df.groupby("puid")["city"]
      .nunique()
)

both_cities = both_cities[both_cities > 1].index

# Удаляем их полностью из обеих групп
df = df[~df["puid"].isin(both_cities)]

In [9]:
# проверяем
df["puid"].duplicated().sum()

0

In [10]:
# посмотрим основные описательные статистики по двум группам
df.groupby("city")["hours"].describe()

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
Москва,5990.0,10.848192,36.925622,0.000022,0.057042,0.888232,5.933439,857.209373
Санкт-Петербург,2306.0,11.264433,39.831755,0.000025,0.060173,0.875355,6.138424,978.764775


В выборке осталось 5990 пользователей из Москвы и 2306 пользователей из Санкт-Петербурга. Изначально 244 пользователя встречались в обеих группах, поэтому для обеспечения независимости выборок мы исключили этих пользователей из обеих групп. После очистки пересечений между группами нет.
Среднее количество часов немного выше у пользователей Санкт-Петербурга — 11,26 против 10,84 в Москве. Медианное значение  немного выше в Мск — 0,89 против 0,88. В обеих группах наблюдается большой разброс значений, а среднее значительно превышает медиану, что указывает на правостороннюю асимметрию распределения и наличие пользователей с очень большим количеством часов прослушивания.

##  3. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [11]:
# односторонний t-тест для двух независимых выборок

moscow = df[df["city"] == "Москва"]["hours"]
spb = df[df["city"] == "Санкт-Петербург"]["hours"]

stat, p_value = ttest_ind(
    moscow,
    spb,
    equal_var=False,
    alternative="less"
)

alpha = 0.05

print(f"p-value: {p_value:.4f}")

if p_value < alpha:
    print("Разница статистически значима")
else:
    print("Разница статистически незначима")

p-value: 0.3318
Разница статистически незначима


## 4. Аналитическая записка
Для сравнения пользователей из Москвы и Санкт-Петербурга использовали **односторонний t-тест**, так как он требовался по условию.

Уровень статистической значимости — **0,05**.

По результатам теста получили **p-value = 0,3318**. Поскольку это значение значительно больше 0,05, **статистически значимых различий между группами не обнаружено**.

Возможные причины такого результата: Наблюдаемая разница между группами может быть связана со случайными колебаниями выборки. Кроме того, на результаты могут влиять отдельные пользователи с очень большим количеством часов прослушивания, поскольку распределение показателя имеет выраженную правостороннюю асимметрию.


# Часть 2. Анализ результатов A/B-тестирования

## 1. Цель и задачи исследования.



### Цель проекта

Оценить результаты A/B-теста новой версии сайта BitMotion Kit и определить, увеличивает ли новый интерфейс конверсию зарегистрированных пользователей в покупку **в течение 7 дней после регистрации** не менее чем на **3 процентных пункта** по сравнению с контрольной группой.


### Задачи

* Проверить корректность проведения A/B-теста.
* Сравнить контрольную и тестовую группы.
* Рассчитать конверсию в покупку.
* Провести статистический тест.
* Сделать вывод о результатах эксперимента.


### Данные
https://code.s3.yandex.net/datasets/ab_test_participants.csv — таблица участников тестов.  
Структура файла:  
user_id — идентификатор пользователя;  
group — группа пользователя;  
ab_test — название теста;  
device — устройство, с которого происходила регистрация.  

https://code.s3.yandex.net/datasets/ab_test_events.zip — архив с одним csv-файлом, в котором собраны события 2020 года;  
Структура файла:  
user_id — идентификатор пользователя;  
event_dt — дата и время события;  
event_name — тип события;  
details — дополнительные данные о событии.

## 2. Загрузка данных, оценка их целостности.


In [12]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [13]:
participants.head()

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


In [14]:
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


In [15]:
events.head()

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [16]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


Таблица participants: данные полные, пропусков нет, удалены данные другого a/b теста, дубликатов нет, группы A и B распределены равномерно  
Таблица events: пропуски есть в details, но они не будут влиять на дальнейшее тестирование. данные в столбце event_dt приведены к типу datetime64. Можно продолжать тестирование

## 3. По таблице `participants` оценка корректности проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [17]:
# проверим название тестов
participants['ab_test'].value_counts()

interface_eu_test          10850
recommender_system_test     3675
Name: ab_test, dtype: int64

In [18]:
# пользователи, участвующие одновременно в двух тестовых группах
# пользователи нашего теста
interface_users = set(
    participants[participants["ab_test"] == "interface_eu_test"]["user_id"]
)

# пользователи конкурирующего теста
recommender_users = set(
    participants[participants["ab_test"] == "recommender_system_test"]["user_id"]
)

# пересечение
overlap = interface_users & recommender_users

print("Количество пересекающихся пользователей:", len(overlap))

Количество пересекающихся пользователей: 887


In [19]:
# удалим 887 пользователей для дальнейшего тестирования
interface_users = set(
    participants[participants["ab_test"] == "interface_eu_test"]["user_id"]
)

recommender_users = set(
    participants[participants["ab_test"] == "recommender_system_test"]["user_id"]
)

overlap = interface_users & recommender_users

# удалим пересекающихся пользователей из participants
participants = participants[
    ~participants["user_id"].isin(overlap)
]

In [20]:
# оставим данные только для interface_eu_test
participants = participants[
    participants["ab_test"] == "interface_eu_test"
]

In [21]:
# проверим на дубликаты
participants["user_id"].duplicated().sum()

0

In [22]:
# посмотрим распределение по группам 
participants['group'].value_counts()

B    5011
A    4952
Name: group, dtype: int64

In [23]:
participants['group'].value_counts(normalize=True) * 100

B    50.296096
A    49.703904
Name: group, dtype: float64

Для анализа были выделены пользователи теста interface_eu_test. Дубликатов пользователей и случаев одновременного участия в группах A и B не обнаружено. Распределение по группам равномерное: 49,7% пользователей относятся к группе A и 50,2% — к группе B. При проверке пересечений с конкурирующим тестом recommender_system_test обнаружено 887 общих пользователей. Они были удалены из обоих тестов, тк другой тест может влиять на результат в рассматриваемом тесте.

3\.2 Проанализируйте данные о пользовательской активности по таблице `events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [24]:
events['event_name'].value_counts()

login                                 248285
product_page                          195606
registration                          144183
purchase                              104836
product_cart                           94373
Start of CIS New Year Gift Lottery         1
Start of Christmas&New Year Promo          1
End of Black Friday Ads Campaign           1
Name: event_name, dtype: int64

In [25]:
# создадим таблицу с датой регистрации
registrations = (
    events[events["event_name"] == "registration"]
    .groupby("user_id")["event_dt"]
    .min()
    .reset_index()
)

In [26]:
# создадим таблицу с датой покупки
purchases = (
    events[events["event_name"] == "purchase"]
    .groupby("user_id")["event_dt"]
    .min()
    .reset_index()
)

In [27]:
# объединим таблицы с таблицей с участниками тестирования
df_test = participants.merge(
    registrations,
    on="user_id",
    how="left"
)

df_test = df_test.merge(
    purchases,
    on="user_id",
    how="left",
    suffixes=("_registration", "_purchase")
)
df_test

,user_id,group,ab_test,device,event_dt_registration,event_dt_purchase
0,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:37:31,NaT
1,001E72F50D1C48FA,A,interface_eu_test,Mac,2020-12-17 15:44:05,NaT
2,002412F1EB3F6E38,B,interface_eu_test,Mac,2020-12-09 09:36:50,NaT
3,002540BE89C930FB,B,interface_eu_test,Android,2020-12-08 18:06:07,NaT
4,0031F1B5E9FBF708,A,interface_eu_test,Android,2020-12-14 00:47:10,2020-12-15 18:54:51
...,...,...,...,...,...,...
9958,FFE5B14BD55C1C5C,A,interface_eu_test,PC,2020-12-19 06:27:24,NaT
9959,FFE600EEC4BA7685,B,interface_eu_test,Android,2020-12-13 21:43:31,NaT
9960,FFE7FC140521F5F6,A,interface_eu_test,PC,2020-12-23 09:10:16,2020-12-26 14:37:21
9961,FFEFC0E55C1CCD4F,A,interface_eu_test,PC,2020-12-13 23:52:15,NaT


In [28]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9963 entries, 0 to 9962
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   user_id                9963 non-null   object        
 1   group                  9963 non-null   object        
 2   ab_test                9963 non-null   object        
 3   device                 9963 non-null   object        
 4   event_dt_registration  9963 non-null   datetime64[ns]
 5   event_dt_purchase      3420 non-null   datetime64[ns]
dtypes: datetime64[ns](2), object(4)
memory usage: 544.9+ KB


- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [29]:
# найдем разницу в днях между регистрацией и покупкой и отберем только те покупки, которые были совершены в течение 7 дней после регистрации
df_test["purchase_7_days"] = (
    (df_test["event_dt_purchase"] >= df_test["event_dt_registration"]) &
    (df_test["event_dt_purchase"] <=
     df_test["event_dt_registration"] + pd.Timedelta(days=7))
).astype(int)

In [30]:
# посмотрим сколько таких пользователей в каждой группе 
df_test.groupby("group")["purchase_7_days"].sum()

group
A    1377
B    1480
Name: purchase_7_days, dtype: int64

Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [31]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p1 = 0.30
p2 = 0.33

alpha = 0.05
power = 0.80

effect_size = proportion_effectsize(p1, p2)

sample_size = NormalIndPower().solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1
)

print("Необходимый размер одной группы:", round(sample_size))
print("Необходимый размер выборки:", round(sample_size * 2))

Необходимый размер одной группы: 3762
Необходимый размер выборки: 7523


- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [32]:
result = (
    df_test.groupby('group')['purchase_7_days']
    .agg(['sum', 'count'])
    .rename(columns={
        'sum': 'buyers',
        'count': 'total_users'
    })
)

result

,buyers,total_users
group,,
A,1377,4952
B,1480,5011


In [33]:
conversion = (
    df_test.groupby('group')['purchase_7_days']
    .mean() * 100
).round(2)

conversion

group
A    27.81
B    29.54
Name: purchase_7_days, dtype: float64

In [34]:
round((conversion['B'] - conversion['A']),2)

1.73

В тестовой группе B конверсия пользователей в покупку в течение 7 дней составила 29,54%, что выше показателя контрольной группы A (27,81%) на 1,73 процентного пункта. Таким образом, в тестовой группе наблюдается положительное изменение пользовательской активности. При этом прирост конверсии пока ниже целевого значения в 3 п.п., поэтому для оценки статистической значимости и итоговой эффективности нового интерфейса необходимо провести статистический тест.

## 4. Оценка результатов A/B-тестирования и выводы

H₀: конверсия в группе B не выше конверсии в группе A.

H₁: конверсия в группе B выше конверсии в группе A.

In [35]:
# для сравнения двух конверсий — долей пользователей, совершивших покупку в течение 7 дней, в двух независимых группах A и B проведем z-тест.
from statsmodels.stats.proportion import proportions_ztest

# количество пользователей с покупкой в течение 7 дней
successes = df_test.groupby("group")["purchase_7_days"].sum()

# общее количество пользователей в каждой группе
nobs = df_test.groupby("group")["purchase_7_days"].count()

z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=nobs,
    alternative="smaller"
)

alpha = 0.05

print(f"p-value: {p_value:.4f}")

if p_value < alpha:
    print("Разница статистически значима")
else:
    print("Разница статистически незначима")

p-value: 0.0283
Разница статистически значима


In [36]:
conversion = df_test.groupby("group")["purchase_7_days"].mean()

increase = (conversion["B"] - conversion["A"]) * 100

print(f"Прирост конверсии: {increase:.2f} п.п.")

if increase >= 3:
    print("Целевой прирост в 3 п.п. достигнут")
else:
    print("Целевой прирост в 3 п.п. не достигнут")

Прирост конверсии: 1.73 п.п.
Целевой прирост в 3 п.п. не достигнут


### Вывод по результатам A/B-теста

Конверсия в покупку в течение 7 дней в тестовой группе B составила **29,54%**, тогда как в контрольной группе A — **27,81%**. Прирост конверсии составил **1,73 процентного пункта**.

По результатам Z-теста разница между группами статистически значима: **p-value = 0,0283**, что меньше уровня значимости 0,05. Следовательно, наблюдаемый рост конверсии статистически значим.

Однако целевой показатель теста — увеличение конверсии **не менее чем на 3 п.п.** — не достигнут. Таким образом, новый интерфейс действительно привёл к статистически значимому росту конверсии, но требуемый бизнес-эффект не был достигнут.
